# Part 3: Email Data

---

### Install Python packages (pip only)

In [1]:
%pip install networkx numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import Python packages

In [2]:
import csv
import networkx as nx
import numpy as np
import operator
from datetime import datetime

---

##### Examine the file "email_data.csv" which represents email behaviour between individuals. Assume that emails are to one recipient only. Each row has three columns, representing a single event where a person (person_a) emailed another person (person_b) on some date and time of day (timestamp). 

##### From this, answer the following questions:

##### Q1. Build a suitable network to represent social connections based on the email behaviour that took place up to and including the first day of May. In doing so, assume that one or more emails from one person to another represents a mutual underlying social connection (i.e., regardless of whether person_a emailed person_b, vice versa, or both). 

##### What non-existent edges would need to be added to the network to connect it so that a hypothetical message could pass along a path from any individual to any other? In doing so aim to minimise the longest shortest path as a result. Do not add these edges to the network for the later questions.

In [3]:
# Build undirected network up to and including May 1st - from GitHub commits notebook
cutoff = datetime(2024, 5, 1, 23, 59, 59)
G1 = nx.Graph()

with open("email_data.csv") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        ts = datetime.strptime(row[2], "%d/%m/%Y %H:%M:%S")
        if ts <= cutoff:
            G1.add_edge(row[0], row[1])

print(f"Nodes: {G1.number_of_nodes()}, Edges: {G1.number_of_edges()}")
print(f"Connected: {nx.is_connected(G1)}")

# Find connected components 
components = sorted(nx.connected_components(G1), key=len, reverse=True)
print(f"Number of components: {len(components)}")
for i, comp in enumerate(components):
    print(f"  Component {i+1}: {len(comp)} nodes")

# Find center of giant component 
giant = G1.subgraph(components[0]).copy()
center_nodes = nx.center(giant)
print(f"\nCenter of giant component: {center_nodes}")
print(f"Diameter of giant component: {nx.diameter(giant)}")

# Connect each small component to the center of the giant
proposed_edges = []
for comp in components[1:]:
    small_node = list(comp)[0]
    proposed_edges.append((small_node, center_nodes[0]))
    print(f"Proposed edge: {small_node} -- {center_nodes[0]}")

# Verify result without modifying G1
G1_test = G1.copy()
G1_test.add_edges_from(proposed_edges)
print(f"\nAfter adding {len(proposed_edges)} edges:")
print(f"Connected: {nx.is_connected(G1_test)}")
print(f"Diameter: {nx.diameter(G1_test)}")


Nodes: 515, Edges: 1643
Connected: False
Number of components: 4
  Component 1: 509 nodes
  Component 2: 2 nodes
  Component 3: 2 nodes
  Component 4: 2 nodes

Center of giant component: ['1253', '1862', '729', '202', '1678', '1776', '1422', '1827', '1320', '1854', '1384', '148']
Diameter of giant component: 7
Proposed edge: 785 -- 1253
Proposed edge: 793 -- 1253
Proposed edge: 921 -- 1253

After adding 3 edges:
Connected: True
Diameter: 7


##### Q2. Build a suitable network to represent social connections based on the cumulation of all email behaviour in the dataset. In doing so, assume that one or emails from one person to another represents a mutual underlying social connection (i.e., regardless of whether person_a emailed person_b, vice versa, or both).

##### Can the social phenomenon, ‘Triadic Closure’, be supported for the set of individuals that exist in both the network created from data up to and including the first day of May (i.e., from Q1) and the network built from all data?

In [4]:
# Build undirected network from all data
G2 = nx.Graph()

with open("email_data.csv") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        G2.add_edge(row[0], row[1])

print(f"G2 Nodes: {G2.number_of_nodes()}, Edges: {G2.number_of_edges()}")
print(f"Connected: {nx.is_connected(G2)}")

# Find shared nodes between G1 and G2
shared_nodes = set(G1.nodes()) & set(G2.nodes())
print(f"Shared nodes: {len(shared_nodes)}")

# Create induced subgraphs for shared nodes
G1_shared = G1.subgraph(shared_nodes).copy()
G2_shared = G2.subgraph(shared_nodes).copy()

# Triadic closure measured by transitivity
t1 = nx.transitivity(G1_shared)
t2 = nx.transitivity(G2_shared)
cc1 = nx.average_clustering(G1_shared)
cc2 = nx.average_clustering(G2_shared)

print(f"\nTransitivity  - G1 shared: {t1:.4f},  G2 shared: {t2:.4f}")
print(f"Avg Clustering - G1 shared: {cc1:.4f}, G2 shared: {cc2:.4f}")
print(f"Triadic closure supported: {t2 > t1}")


G2 Nodes: 1827, Edges: 12600
Connected: False
Shared nodes: 515

Transitivity  - G1 shared: 0.0456,  G2 shared: 0.0834
Avg Clustering - G1 shared: 0.0573, G2 shared: 0.1223
Triadic closure supported: True


##### Q3. Using the largest connect connected components from both networks, is the average and median maximum degree of separation between individuals and all others larger in the network built from all data (i.e., from Q2) than the network built from data up to and including the first day of May (i.e., from Q1)?

In [5]:
# Extract largest connected components
LCC1 = G1.subgraph(max(nx.connected_components(G1), key=len)).copy()
LCC2 = G2.subgraph(max(nx.connected_components(G2), key=len)).copy()

print(f"LCC1: {LCC1.number_of_nodes()} nodes, {LCC1.number_of_edges()} edges")
print(f"LCC2: {LCC2.number_of_nodes()} nodes, {LCC2.number_of_edges()} edges")

# Eccentricity = max shortest path from each node
ecc1 = list(nx.eccentricity(LCC1).values())
ecc2 = list(nx.eccentricity(LCC2).values())

print(f"\nLCC1 eccentricity - Mean: {np.mean(ecc1):.2f}, Median: {np.median(ecc1):.2f}")
print(f"LCC2 eccentricity - Mean: {np.mean(ecc2):.2f}, Median: {np.median(ecc2):.2f}")
print(f"LCC1 diameter: {np.max(ecc1)}, LCC2 diameter: {np.max(ecc2)}")

LCC1: 509 nodes, 1640 edges
LCC2: 1815 nodes, 12594 edges

LCC1 eccentricity - Mean: 5.38, Median: 5.00
LCC2 eccentricity - Mean: 5.70, Median: 6.00
LCC1 diameter: 7, LCC2 diameter: 8


##### Q4. Using the largest connected component of network constructed from all data (i.e., from Q2), assume the role of an outsider with complete visibility of the network that now wishes to spread a hypothetical email such that everyone in the component would know the information it contained as quickly as possible. Assume that the information will spread in sequential timesteps using the following mechanism. If an individual is told the information in an email at timestep 𝑡, the individual will forward the email to all of their direct connections at timestep 𝑡+1. Individuals can therefore be told the information more than once.

##### If you could only select 1 individual to tell at timestep 0, what individuals could you select which would result in the message being received by everyone in the fewest timesteps as possible and what would the number of timesteps be?

In [6]:
# Spreading = BFS: timesteps to reach all = eccentricity of starting node
# Best single node = minimum eccentricity (center node) - from Twitch notebook
ecc2 = nx.eccentricity(LCC2)
min_ecc = min(ecc2.values())
center_nodes = [n for n, e in ecc2.items() if e == min_ecc]

print(f"Minimum timesteps to reach everyone: {min_ecc}")
print(f"Number of optimal starting nodes: {len(center_nodes)}")
print(f"Optimal starting nodes: {center_nodes}")

Minimum timesteps to reach everyone: 4
Number of optimal starting nodes: 2
Optimal starting nodes: ['1049', '1320']


##### Q5. If you had to select any 5 individuals to tell at timestep 0, is there an example set of individuals that would result in the information being received by everyone in fewer timesteps than the single individual selection in Q4? In determining your answer, use one or more appropriate network connectivity measures, rather than an exhaustive search through every combination of individuals.

In [7]:
# Use betweenness centrality to identify well-connected bridge nodes
bc = sorted(nx.betweenness_centrality(LCC2).items(), key=operator.itemgetter(1), reverse=True)

print("Top 5 nodes by betweenness centrality:")
for n, v in bc[:5]:
    print(f"  Node {n}: {v:.4f}")

top5_bc = [n for n, v in bc[:5]]

# Simulate multi-source BFS from 5 nodes simultaneously
def multi_source_timesteps(G, sources):
    visited = set(sources)
    frontier = set(sources)
    t = 0
    while len(visited) < G.number_of_nodes():
        next_frontier = set()
        for node in frontier:
            for neighbour in G.neighbors(node):
                if neighbour not in visited:
                    next_frontier.add(neighbour)
        visited |= next_frontier
        frontier = next_frontier
        t += 1
        if not frontier:
            break
    return t

ts_5 = multi_source_timesteps(LCC2, top5_bc)
print(f"\nTimesteps with top 5 betweenness nodes: {ts_5}")
print(f"Timesteps with best single node (Q4):   {min_ecc}")
print(f"Improvement over single node: {min_ecc - ts_5} timesteps")

Top 5 nodes by betweenness centrality:
  Node 148: 0.0595
  Node 1320: 0.0549
  Node 1049: 0.0544
  Node 1639: 0.0514
  Node 202: 0.0508

Timesteps with top 5 betweenness nodes: 4
Timesteps with best single node (Q4):   4
Improvement over single node: 0 timesteps
